# Giai đoạn 0 — Thẩm định cốt lõi AI (CatVTON)

**Mục tiêu:** chạy thử CatVTON trên ~10 ảnh người & 10 ảnh áo "trần trụi" (chưa can thiệp thêm), đo **tốc độ** và **chất lượng** để quyết định go/no-go trước khi tách việc sang Giai đoạn 1.

**Trước khi chạy:**
- Bật GPU: `Settings → Accelerator → GPU T4 x2` (hoặc P100).
- Bật Internet: `Settings → Internet → On` (cần tải checkpoint từ HuggingFace, ~vài GB, chỉ tải 1 lần).
- Tạo 1 Kaggle Dataset chứa 2 thư mục `person/` (10 ảnh người) và `cloth/` (10 ảnh áo), rồi Add data vào notebook. Nếu chạy trên Colab, đổi `PERSON_DIR`/`CLOTH_DIR` sang đường dẫn Google Drive tương ứng.

Notebook này dùng thẳng pipeline gốc của CatVTON (`CatVTONPipeline` + `AutoMasker`) — không qua giao diện Gradio — để dễ chạy hàng loạt và ghi log thời gian.


In [3]:
#1
!nvidia-smi

Fri Aug 28 00:54:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Cài đặt

In [4]:
#2
import os
os.chdir("/kaggle/working")

if not os.path.isdir("/kaggle/working/CatVTON"):
    !git clone https://github.com/Zheng-Chong/CatVTON.git

%cd /kaggle/working/CatVTON
!grep -vi '^gradio' requirements.txt > requirements_notebook.txt
!pip install -q -r requirements_notebook.txt
!pip install -q huggingface_hub
# fvcore/iopath/yacs không nằm trong requirements.txt của repo nhưng detectron2 (được
# localize trong thư mục detectron2/ của repo) cần chúng để import được — cài tay:
!pip install -q fvcore iopath yacs pycocotools omegaconf cloudpickle av

# Kaggle hiện cài sẵn 1 bản PyTorch build cho sm_70 trở lên (đã bỏ hỗ trợ kiến trúc
# Pascal / sm_60 — đúng GPU P100). Ép cài lại bản cũ hơn, build CUDA 12.1, vẫn còn
# kernel cho sm_60, để tránh lỗi "no kernel image is available for execution":
# DÒNG MỚI
!pip install -q --force-reinstall torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu121
print("\n>>> Cài xong. BẮT BUỘC Restart kernel (Kernel > Restart & Clear Output) rồi chạy lại từ đầu trước khi qua bước tiếp theo.")

/kaggle/working/CatVTON
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
ERROR: Cannot install -r requirements_notebook.txt (line 14), -r requirements_notebook.txt (line 3), -r requirements_notebook.txt (line 4) and huggingface_hub<2.0 and >=0.34.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0+cu121, 2.2.1+cu121, 2.2.2+cu121, 2.3.0+cu121, 2.3.1+cu121, 2.4.0+cu121, 2.4.1+cu121, 2.5.0+cu121, 2.5.1+cu121)
ERROR: No matching distribution found for torch==2.1.2

>>> Cài xong. BẮT BUỘC Restart kerne

## 2. Tải checkpoint & khởi tạo pipeline

Lần chạy đầu sẽ tải checkpoint CatVTON (`zhengchong/CatVTON`) + base model inpainting từ HuggingFace, có thể mất 5–10 phút tuỳ tốc độ mạng.

**Lưu ý về `mixed_precision`:** GPU Kaggle (T4/P100) không tăng tốc tốt với `bf16` (cần kiến trúc Ampere trở lên). Dùng `fp16` cho Kaggle; khi lên RunPod/Vast.ai thuê GPU đời mới hơn (A100/4090) có thể đổi lại `bf16`.


> **Cập nhật cho T4 x2:** pipeline không còn được dựng ở đây — xem phần "5. Chạy inference hàng loạt" bên dưới, giờ chạy **song song trên cả 2 GPU** bằng 2 tiến trình con độc lập.


In [5]:
#3
# LƯU Ý: cell này giờ CHỈ tải sẵn checkpoint về cache HuggingFace (nhanh nếu đã
# tải rồi). KHÔNG dựng CatVTONPipeline/AutoMasker ở tiến trình chính (kernel)
# nữa — việc đó được chuyển xuống từng worker (1 worker/GPU) ở bước "Chạy
# inference hàng loạt" bên dưới, để mỗi worker tự load model lên đúng 1 GPU,
# tránh trường hợp kernel + worker cùng load model chồng lên GPU 0.
import os
from huggingface_hub import snapshot_download
from utils import init_weight_dtype  # chỉ import để kiểm tra module OK

MIXED_PRECISION = "fp16"  # "fp16" cho T4/P100 (Kaggle) | "bf16" nếu chạy trên GPU Ampere+ (A100, 4090...)

repo_path = snapshot_download(repo_id="zhengchong/CatVTON")
print("Checkpoint đã sẵn sàng trong cache tại:", repo_path)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Checkpoint đã sẵn sàng trong cache tại: /root/.cache/huggingface/hub/models--zhengchong--CatVTON/snapshots/2969fcf85fe62f2036605716f0b56f0b81d01d79


## 3. Cấu hình đường dẫn & tham số

In [6]:
#4
# ==== CHỈNH LẠI CHO ĐÚNG DATASET CỦA BẠN ====
PERSON_DIR = "/kaggle/input/datasets/js042710/10clothes10man/person"   # thư mục chứa ảnh người
CLOTH_DIR  = "/kaggle/input/datasets/js042710/10clothes10man/clothes"    # thư mục chứa ảnh áo
OUTPUT_DIR = "/kaggle/working/outputs"

CLOTH_TYPE_DEFAULT = "upper"   # "upper" | "lower" | "overall" — dùng khi để AutoMasker tự tạo mask
WIDTH, HEIGHT = 768, 1024
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 2.5
SEED = 42                      # đặt -1 nếu muốn random mỗi lần chạy

os.makedirs(OUTPUT_DIR, exist_ok=True)

VALID_EXT = (".jpg", ".jpeg", ".png", ".webp")
person_paths = sorted(
    os.path.join(PERSON_DIR, f) for f in os.listdir(PERSON_DIR) if f.lower().endswith(VALID_EXT)
)
cloth_paths = sorted(
    os.path.join(CLOTH_DIR, f) for f in os.listdir(CLOTH_DIR) if f.lower().endswith(VALID_EXT)
)

print(f"Tìm thấy {len(person_paths)} ảnh người, {len(cloth_paths)} ảnh áo")
assert person_paths and cloth_paths, "Không tìm thấy ảnh — kiểm tra lại PERSON_DIR / CLOTH_DIR"


Tìm thấy 10 ảnh người, 10 ảnh áo


## 4. Ghép cặp thử nghiệm

Mặc định ghép 1-1 theo thứ tự (`person[i]` với `cloth[i]`) → đúng 10 cặp như kế hoạch. Nếu muốn thử nhiều tổ hợp hơn (đánh giá kỹ hơn nhưng tốn thời gian hơn), đổi sang `itertools.product`.

In [7]:
#5
pairs = list(zip(person_paths, cloth_paths))
# Muốn thử mọi tổ hợp (10x10=100 cặp) thay vì 1-1, dùng dòng dưới thay cho dòng trên:
# import itertools; pairs = list(itertools.product(person_paths, cloth_paths))

print(f"Sẽ chạy {len(pairs)} cặp thử nghiệm")

Sẽ chạy 10 cặp thử nghiệm


## 5. Chạy inference hàng loạt + đo thời gian

**Cập nhật T4 x2:** thay vì chạy tuần tự trên 1 GPU, notebook giờ chia danh sách cặp thành 2 phần bằng nhau, giao mỗi phần cho 1 tiến trình con riêng (`infer_worker.py`) — mỗi tiến trình chỉ thấy đúng 1 GPU qua `CUDA_VISIBLE_DEVICES` và chạy hoàn toàn độc lập, không cần đồng bộ/giao tiếp giữa 2 GPU (T4 x2 trên Kaggle không có NVLink nên đây là cách tận dụng hiệu quả nhất — kỳ vọng tổng thời gian giảm gần một nửa).

In [8]:
%%writefile /kaggle/working/CatVTON/infer_worker.py
"""
Worker chạy inference CatVTON cho 1 "shard" (một phần) danh sách cặp
(person, cloth) trên ĐÚNG 1 GPU.

Được gọi từ notebook qua subprocess — mỗi worker là 1 tiến trình Python
riêng biệt, chỉ thấy 1 GPU nhờ biến môi trường CUDA_VISIBLE_DEVICES (đặt ở
notebook trước khi Popen). Nhờ chạy tiến trình riêng (không phải fork/thread)
nên tránh được lỗi "CUDA đã init không fork được" khi dùng multiprocessing
ngay trong kernel Jupyter.
"""
import argparse
import json
import os
import time

import pandas as pd
import torch
from PIL import Image

from diffusers.image_processor import VaeImageProcessor
from huggingface_hub import snapshot_download

from model.cloth_masker import AutoMasker, vis_mask
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--shard", required=True, help="File JSON chứa danh sách cặp của shard này")
    ap.add_argument("--output-dir", required=True)
    ap.add_argument("--cloth-type", default="upper")
    ap.add_argument("--width", type=int, default=768)
    ap.add_argument("--height", type=int, default=1024)
    ap.add_argument("--steps", type=int, default=50)
    ap.add_argument("--guidance", type=float, default=2.5)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--mixed-precision", default="fp16")
    ap.add_argument("--tag", required=True, help="Nhãn shard (vd: '0', '1') dùng để đặt tên file log")
    args = ap.parse_args()

    with open(args.shard) as f:
        shard = json.load(f)  # list of {"pair_index", "person", "cloth"}

    os.makedirs(args.output_dir, exist_ok=True)

    gpu_name = torch.cuda.get_device_name(0)
    print(f"[worker {args.tag}] dùng GPU: {gpu_name}", flush=True)

    repo_path = snapshot_download(repo_id="zhengchong/CatVTON")
    pipeline = CatVTONPipeline(
        base_ckpt="booksforcharlie/stable-diffusion-inpainting",
        attn_ckpt=repo_path,
        attn_ckpt_version="mix",
        weight_dtype=init_weight_dtype(args.mixed_precision),
        use_tf32=True,
        device="cuda",  # trong tiến trình con, "cuda" = GPU được cấp qua CUDA_VISIBLE_DEVICES
    )
    mask_processor = VaeImageProcessor(
        vae_scale_factor=8, do_normalize=False, do_binarize=True, do_convert_grayscale=True
    )
    automasker = AutoMasker(
        densepose_ckpt=os.path.join(repo_path, "DensePose"),
        schp_ckpt=os.path.join(repo_path, "SCHP"),
        device="cuda",
    )
    print(f"[worker {args.tag}] model sẵn sàng, xử lý {len(shard)} cặp.", flush=True)

    records = []
    for n, item in enumerate(shard, start=1):
        i = item["pair_index"]
        person_path, cloth_path = item["person"], item["cloth"]

        person_image = Image.open(person_path).convert("RGB")
        cloth_image = Image.open(cloth_path).convert("RGB")
        person_image = resize_and_crop(person_image, (args.width, args.height))
        cloth_image = resize_and_padding(cloth_image, (args.width, args.height))

        mask = automasker(person_image, args.cloth_type)["mask"]
        mask = mask_processor.blur(mask, blur_factor=9)

        generator = torch.Generator(device="cuda").manual_seed(args.seed) if args.seed != -1 else None

        torch.cuda.synchronize()
        t0 = time.time()
        result_image = pipeline(
            image=person_image,
            condition_image=cloth_image,
            mask=mask,
            num_inference_steps=args.steps,
            guidance_scale=args.guidance,
            generator=generator,
        )[0]
        torch.cuda.synchronize()
        elapsed = time.time() - t0

        out_name = f"pair_{i:02d}.png"
        result_image.save(os.path.join(args.output_dir, out_name))

        masked_person = vis_mask(person_image, mask)
        grid = Image.new("RGB", (args.width * 3, args.height))
        grid.paste(person_image, (0, 0))
        grid.paste(masked_person, (args.width, 0))
        grid.paste(cloth_image, (args.width * 2, 0))
        grid.save(os.path.join(args.output_dir, f"pair_{i:02d}_debug.png"))

        records.append({
            "pair_index": i,
            "person": os.path.basename(person_path),
            "cloth": os.path.basename(cloth_path),
            "seconds": round(elapsed, 2),
            "output_file": out_name,
        })
        print(f"[worker {args.tag} | {n}/{len(shard)}] pair {i}: "
              f"{os.path.basename(person_path)} + {os.path.basename(cloth_path)} -> {elapsed:.2f}s",
              flush=True)

    out_csv = os.path.join(args.output_dir, f"shard_{args.tag}.csv")
    pd.DataFrame(records).to_csv(out_csv, index=False)
    print(f"[worker {args.tag}] xong, log lưu tại {out_csv}", flush=True)


if __name__ == "__main__":
    main()


Writing /kaggle/working/CatVTON/infer_worker.py


In [9]:
#6
import json
import os
import subprocess
import sys
import threading
import time

import pandas as pd

CATVTON_DIR = "/kaggle/working/CatVTON"
SHARD_DIR = os.path.join(OUTPUT_DIR, "shards")
os.makedirs(SHARD_DIR, exist_ok=True)

N_GPUS = 2  # Kaggle T4 x2

# Chia các cặp theo kiểu round-robin (xen kẽ) cho từng GPU, để nếu ảnh có độ
# khó/kích thước khác nhau thì 2 GPU vẫn có tổng thời gian tương đối cân bằng.
shard_lists = {str(g): [] for g in range(N_GPUS)}
for i, (person_path, cloth_path) in enumerate(pairs):
    gpu_key = str(i % N_GPUS)
    shard_lists[gpu_key].append({"pair_index": i, "person": person_path, "cloth": cloth_path})

shard_files = {}
for gpu_key, items in shard_lists.items():
    path = os.path.join(SHARD_DIR, f"shard_{gpu_key}.json")
    with open(path, "w") as f:
        json.dump(items, f)
    shard_files[gpu_key] = path
    print(f"GPU {gpu_key}: {len(items)} cặp -> {path}")

def launch_worker(gpu_key, shard_path):
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = gpu_key  # tiến trình con chỉ thấy đúng 1 GPU vật lý
    cmd = [
        sys.executable, "infer_worker.py",
        "--shard", shard_path,
        "--output-dir", OUTPUT_DIR,
        "--cloth-type", CLOTH_TYPE_DEFAULT,
        "--width", str(WIDTH), "--height", str(HEIGHT),
        "--steps", str(NUM_INFERENCE_STEPS),
        "--guidance", str(GUIDANCE_SCALE),
        "--seed", str(SEED),
        "--mixed-precision", MIXED_PRECISION,
        "--tag", gpu_key,
    ]
    return subprocess.Popen(
        cmd, cwd=CATVTON_DIR, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )

procs = [(gpu_key, launch_worker(gpu_key, path)) for gpu_key, path in shard_files.items()]

def stream_output(gpu_key, proc):
    for line in proc.stdout:
        print(f"[GPU{gpu_key}] {line}", end="")

threads = [threading.Thread(target=stream_output, args=(k, p)) for k, p in procs]
t_start = time.time()
for t in threads:
    t.start()
for _, p in procs:
    p.wait()
for t in threads:
    t.join()
total_wall_time = time.time() - t_start

for gpu_key, p in procs:
    if p.returncode != 0:
        raise RuntimeError(f"Worker GPU {gpu_key} lỗi (return code {p.returncode}) — xem log phía trên.")

print(f"\n>>> Cả 2 GPU chạy xong sau {total_wall_time:.2f}s (wall-clock thực tế).")

# Gộp log của 2 shard lại, sắp xếp theo đúng pair_index ban đầu để các cell
# bên dưới (tổng hợp thời gian, soi kết quả, zip) dùng lại được nguyên vẹn.
shard_dfs = [pd.read_csv(os.path.join(OUTPUT_DIR, f"shard_{g}.csv")) for g in shard_files]
records = (
    pd.concat(shard_dfs, ignore_index=True)
    .sort_values("pair_index")
    .to_dict("records")
)


GPU 0: 5 cặp -> /kaggle/working/outputs/shards/shard_0.json
GPU 1: 5 cặp -> /kaggle/working/outputs/shards/shard_1.json
[GPU0] Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[GPU0] Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[GPU1] Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[GPU1] Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
[GPU0] [worker 0] dùng GPU: Tesla T4
[GPU1] [worker 1] dùng GPU: Tesla T4
[GPU0] 
[GPU0] Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 11224.72it/s]
[GPU0] /usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarn

## 6. Tổng hợp thời gian

In [10]:
#7
df = pd.DataFrame(records)
df.to_csv(os.path.join(OUTPUT_DIR, "timing_log.csv"), index=False)

print(f"Thời gian trung bình / ảnh : {df['seconds'].mean():.2f}s")
print(f"Nhanh nhất                : {df['seconds'].min():.2f}s")
print(f"Chậm nhất                 : {df['seconds'].max():.2f}s")
print(f"Tổng thời gian ({len(df)} ảnh) : {df['seconds'].sum():.2f}s")

df

Thời gian trung bình / ảnh : 115.19s
Nhanh nhất                : 107.21s
Chậm nhất                 : 119.84s
Tổng thời gian (10 ảnh) : 1151.94s


,pair_index,person,cloth,seconds,output_file
0,0,00006_00.jpg,00071_00.jpg,114.83,pair_00.png
1,1,00008_00.jpg,00074_00.jpg,107.21,pair_01.png
2,2,00013_00.jpg,00075_00.jpg,119.00,pair_02.png
3,3,00017_00.jpg,00084_00.jpg,111.93,pair_03.png
4,4,00034_00.jpg,00094_00.jpg,119.55,pair_04.png
5,5,00035_00.jpg,00095_00.jpg,113.05,pair_05.png
6,6,00055_00.jpg,00096_00.jpg,119.55,pair_06.png
7,7,00057_00.jpg,00110_00.jpg,113.56,pair_07.png
8,8,00064_00.jpg,00112_00.jpg,119.84,pair_08.png
9,9,00067_00.jpg,00121_00.jpg,113.42,pair_09.png


## 8. Ghi chú đánh giá & quyết định go/no-go

Sau khi chạy xong, mở toàn bộ ảnh trong `OUTPUT_DIR` (đặc biệt các file `*_debug.png` để soi mask) và tự đánh giá:

- **Biến dạng**: tay áo, cổ áo, viền vải có bị méo/nhòe không?
- **Giữ chất liệu**: hoạ tiết, màu sắc áo gốc có được giữ đúng không?
- **Mask tự động**: `AutoMasker` có bắt đúng vùng cần thay không, hay lem ra ngoài?
- **Tốc độ**: `timing_log.csv` cho số giây/ảnh — dùng để ước tính chi phí GPU thuê ở Giai đoạn 1 (RunPod/Vast.ai tính theo giờ).

**Nếu ổn** → chốt `api_contract.md` với Thành viên B, chuyển sang Giai đoạn 1.
**Nếu chất lượng kém / quá chậm** → cân nhắc đổi sang IDM-VTON hoặc giảm `NUM_INFERENCE_STEPS` trước khi quyết định cuối.

*(Tuỳ chọn: nếu muốn định lượng bằng FID/KID, cần thêm một tập ảnh "ground truth" để so sánh — repo CatVTON có sẵn `eval.py` cho việc này, nhưng không bắt buộc ở Giai đoạn 0.)*


In [12]:
import shutil
from IPython.display import FileLink
shutil.make_archive('output', 'zip', '/kaggle/working')
FileLink(r'output.zip')

/kaggle/working/CatVTON/output.zip